Some Materials in /Work_bio/references/Metagenomics

shotgun meta genomics 组装为基础的和不组装的 nf-core pipeline 分别是什么？ 请解释原理和怎么用

在鸟枪法宏基因组（Shotgun Metagenomics）分析中，基于组装和不基于组装是两种截然不同的核心分析策略。
在 nf-core 社区中，这两条路径分别由两个高度集成的最佳实践管道（Pipeline）所代表：

基于组装（Assembly-based）的管道：nf-core/mag（Metagenome-Assembled Genomes）不基于组装（Assembly-free / Read-based）的管道：nf-core/taxprofiler

一、 原理详解🧬 1. nf-core/mag（基于组装与分箱）核心逻辑：将测序得到的短片段（Reads）像拼图一样拼接成较长的连续序列（Contigs）。接着，根据 Contigs 在不同样本中的丰度变化模式（Co-abundance）以及自身核苷酸组成特征（如四核苷酸频率 TNF），将属于同一种生物的 Contigs 聚类归纳到不同的“箱子”里，从而重构出接近完整的单菌基因组（即 MAGs）。主要流程步骤：质控与去宿主：使用 fastp 过滤低质量序列，并去除宿主（如人或特定模式动物）的 DNA 污染。宏基因组组装：调用 MEGAHIT 或 SPAdes（支持短读长和长读长混合组装）将 Reads 拼接为 Contigs。基因预测：用 Prodigal 在 Contigs 上预测蛋白质编码基因。分箱（Binning）：同时使用 MetaBAT2、MaxBin2、SemiBin2 等多个分箱软件，将 Contigs 聚类为不同的 Bins（潜在的菌株基因组）。质控与提纯：利用 CheckM/CheckM2 评估 Bins 的完整度（Completeness）和污染度（Contamination），并通过 DAS Tool 对不同软件的分箱结果进行集成优化。分类学鉴定：使用 GTDB-Tk 对高质量的 MAG 进行精确的物种分类学指派。📊 2. nf-core/taxprofiler（不基于组装 / 基于 Reads）核心逻辑：跳过极其消耗资源的序列拼接步骤，直接将清洗后的每一条原始 Read 与已知的物种参考数据库进行比对。通过计算比对到各个物种上的 Read 数量，直接推算出样本中微生物的物种组成、分类学层级以及相对丰度。主要流程步骤：质控与预处理：使用 fastp 或 AdapterRemoval 过滤低质量 Reads 并去除接头。多工具并行分类：这是该管道的核心优势，它支持并行化运行多种主流的分类学定量工具。包括基于 $k$-mer 的 Kraken2 / Bracken、基于 Marker 基因的 MetaPhlAn、基于全基因组比对的 Centrifuge、基于蛋白质比对的 Kaiju 以及进行功能和物种联合分析的 HUMAnN3。结果标准化：将不同工具输出的格式迥异的丰度表，转化为标准统一的输出格式（如通用丰度矩阵），并使用 MultiQC 统一生成可视化报告。